### 2.1 Geographic Patterns: Area vs Financial Variables

We investigate whether the geographic area (Nord, Centro, Sud/Isole) is associated with differences in wealth, income, job composition, and debt levels. This is relevant because Italian macro-regions have well-known socio-economic disparities.

In [ ]:
# ── Area vs Wealth, Income, Job, Debt ─────────────────────────────────────────
area_map = {1: 'Nord', 2: 'Centro', 3: 'Sud/Isole'}
job_map  = {1: 'Unemployed', 2: 'Employee', 3: 'Manager', 4: 'Entrepreneur', 5: 'Retired'}
df['Area_L'] = df['Area'].map(area_map)
df['Job_L']  = df['Job'].map(job_map)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Geographic Area vs Financial Variables', fontsize=16, fontweight='bold')

# Area vs Wealth — boxplot
sns.boxplot(data=df, x='Area_L', y='Wealth', hue='Area_L', ax=axes[0,0],
            palette='Blues', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[0,0].set_title('Area vs Wealth')
axes[0,0].set_xlabel('')
axes[0,0].set_ylabel('Wealth (percentile)')

# Area vs Income — boxplot
sns.boxplot(data=df, x='Area_L', y='Income', hue='Area_L', ax=axes[0,1],
            palette='Oranges', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[0,1].set_title('Area vs Income')
axes[0,1].set_xlabel('')
axes[0,1].set_ylabel('Income (percentile)')

# Area vs Job — stacked bar (proportions)
job_area_ct = pd.crosstab(df['Area_L'], df['Job_L'], normalize='index')
job_area_ct = job_area_ct[['Unemployed','Employee','Manager','Entrepreneur','Retired']]
job_area_ct.loc[['Nord','Centro','Sud/Isole']].plot(
    kind='bar', stacked=True, ax=axes[1,0], colormap='tab10', edgecolor='white', linewidth=0.5)
axes[1,0].set_title('Area vs Job Distribution')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('Proportion')
axes[1,0].legend(fontsize=8, loc='upper right')
axes[1,0].tick_params(axis='x', rotation=0)

# Area vs Debt — boxplot
sns.boxplot(data=df, x='Area_L', y='Debt', hue='Area_L', ax=axes[1,1],
            palette='Reds', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[1,1].set_title('Area vs Debt')
axes[1,1].set_xlabel('')
axes[1,1].set_ylabel('Debt (percentile)')

plt.tight_layout()
plt.show()

# Print summary stats
print('--- Median values by Area ---')
print(df.groupby('Area_L')[['Income','Wealth','Debt']].median().round(3).to_string())

### 2.2 Investment Behavior: Investments vs Financial Education & Digital Propensity

Are clients who invest (especially in capital accumulation plans) more financially educated and more digitally active? This relationship is central to understanding the drivers of investment behavior.

In [ ]:
# ── Investments vs FinEdu and Digital ─────────────────────────────────────────
inv_map = {1: 'No investments', 2: 'Lump Sum', 3: 'Capital Acc. (PAC)'}
df['Inv_L'] = df['Investments'].map(inv_map)
inv_order = ['No investments', 'Lump Sum', 'Capital Acc. (PAC)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Investment Type vs Financial Sophistication', fontsize=16, fontweight='bold')

# Investments vs FinEdu — violin plot (shows full distribution shape)
sns.violinplot(data=df, x='Inv_L', y='FinEdu', hue='Inv_L', ax=axes[0],
               palette=['#E8A87C','#D5CABD','#41B3A3'], order=inv_order,
               inner='quartile', cut=0, legend=False)
axes[0].set_title('Investments vs Financial Education')
axes[0].set_xlabel('')
axes[0].set_ylabel('Financial Education (percentile)')

# Investments vs Digital — violin plot
sns.violinplot(data=df, x='Inv_L', y='Digital', hue='Inv_L', ax=axes[1],
               palette=['#E8A87C','#D5CABD','#41B3A3'], order=inv_order,
               inner='quartile', cut=0, legend=False)
axes[1].set_title('Investments vs Digital Propensity')
axes[1].set_xlabel('')
axes[1].set_ylabel('Digital (percentile)')

plt.tight_layout()
plt.show()

print('--- Mean FinEdu & Digital by Investment Type ---')
print(df.groupby('Inv_L')[['FinEdu','Digital']].mean().round(3).loc[inv_order].to_string())

### 2.3 Investment Behavior: Investments vs Age & Job

Do older or retired clients prefer different investment types? Does professional status influence investment choice? Understanding the demographic drivers of investment behavior helps contextualize the clustering results.

In [ ]:
# ── Investments vs Age and Job ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Investment Type vs Demographics', fontsize=16, fontweight='bold')

# Investments vs Age — boxplot
sns.boxplot(data=df, x='Inv_L', y='Age', hue='Inv_L', ax=axes[0],
            palette=['#4C72B0','#DD8452','#55A868'], order=inv_order, legend=False)
axes[0].set_title('Investment Type vs Age')
axes[0].set_xlabel('')
axes[0].set_ylabel('Age (years)')

# Investments vs Job — heatmap of proportions
inv_job_ct = pd.crosstab(df['Inv_L'], df['Job_L'], normalize='index') * 100
inv_job_ct = inv_job_ct.loc[inv_order, ['Unemployed','Employee','Manager','Entrepreneur','Retired']]

sns.heatmap(inv_job_ct, annot=True, fmt='.1f', cmap='YlGnBu', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '% within investment type'})
axes[1].set_title('Investment Type vs Job (row %)')
axes[1].set_xlabel('')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('--- Median Age by Investment Type ---')
print(df.groupby('Inv_L')['Age'].median().loc[inv_order].to_string())

### 2.4 Investment Behavior: Investments vs Wealth & Saving Propensity

We expect wealthier clients and stronger savers to invest more actively. A scatter plot lets us see the joint distribution and how investment types cluster in the Wealth-Saving space.

In [ ]:
# ── Investments vs Wealth and Saving ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('Investment Type vs Wealth & Saving', fontsize=16, fontweight='bold')

inv_colors = {'No investments': '#E07A5F', 'Lump Sum': '#81B29A', 'Capital Acc. (PAC)': '#3D405B'}

# Scatter: Wealth vs Saving, colored by investment type
for inv_type in inv_order:
    mask = df['Inv_L'] == inv_type
    axes[0].scatter(df.loc[mask, 'Wealth'], df.loc[mask, 'Saving'],
                    alpha=0.15, s=8, label=inv_type, color=inv_colors[inv_type])
axes[0].set_xlabel('Wealth (percentile)')
axes[0].set_ylabel('Saving (percentile)')
axes[0].set_title('Wealth vs Saving by Investment Type')
axes[0].legend(fontsize=9, markerscale=4)
axes[0].grid(True, alpha=0.2)

# Boxplot side-by-side: Wealth and Saving by Investment type
df_melted = df[['Inv_L','Wealth','Saving']].melt(
    id_vars='Inv_L', var_name='Variable', value_name='Value')
sns.boxplot(data=df_melted, x='Inv_L', y='Value', hue='Variable',
            ax=axes[1], palette=['#3D405B','#81B29A'], order=inv_order)
axes[1].set_title('Wealth & Saving Distribution by Investment Type')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentile')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('--- Mean Wealth & Saving by Investment Type ---')
print(df.groupby('Inv_L')[['Wealth','Saving']].mean().round(3).loc[inv_order].to_string())

### 2.5 Family Dynamics: FamilySize vs Debt & Saving

Larger families might carry more debt and save less. This relationship helps explain the financial stress profile of different household sizes.

In [ ]:
# ── FamilySize vs Debt and Saving ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Family Size vs Financial Behavior', fontsize=16, fontweight='bold')

# FamilySize vs Debt — boxplot
sns.boxplot(data=df, x='FamilySize', y='Debt', hue='FamilySize', ax=axes[0],
            palette='rocket_r', legend=False)
axes[0].set_title('Family Size vs Debt')
axes[0].set_xlabel('Family Size (members)')
axes[0].set_ylabel('Debt (percentile)')

# FamilySize vs Saving — boxplot
sns.boxplot(data=df, x='FamilySize', y='Saving', hue='FamilySize', ax=axes[1],
            palette='crest', legend=False)
axes[1].set_title('Family Size vs Saving Propensity')
axes[1].set_xlabel('Family Size (members)')
axes[1].set_ylabel('Saving (percentile)')

plt.tight_layout()
plt.show()

# Mean + count per FamilySize
fam_stats = df.groupby('FamilySize').agg(
    n_clients=('Debt', 'count'),
    avg_debt=('Debt', 'mean'),
    avg_saving=('Saving', 'mean')
).round(3)
print('--- Debt & Saving by Family Size ---')
print(fam_stats.to_string())

### 2.6 ESG Propensity: ESG vs Age & Wealth

ESG (Environmental, Social, Governance) awareness is a growing dimension in financial services. Is it driven by age (younger generations more ESG-conscious?) or by wealth (richer clients can afford ESG preferences?)?

In [ ]:
# ── ESG vs Age and Wealth ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('ESG Propensity vs Age & Wealth', fontsize=16, fontweight='bold')

# ESG vs Age — scatter with density coloring
hb1 = axes[0].hexbin(df['Age'], df['ESG'], gridsize=25, cmap='YlOrRd', mincnt=1)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('ESG Propensity (percentile)')
axes[0].set_title('ESG vs Age')
plt.colorbar(hb1, ax=axes[0], label='Count')

# ESG vs Wealth — scatter with density coloring
hb2 = axes[1].hexbin(df['Wealth'], df['ESG'], gridsize=25, cmap='YlGnBu', mincnt=1)
axes[1].set_xlabel('Wealth (percentile)')
axes[1].set_ylabel('ESG Propensity (percentile)')
axes[1].set_title('ESG vs Wealth')
plt.colorbar(hb2, ax=axes[1], label='Count')

plt.tight_layout()
plt.show()

# Correlation
print('--- Correlations with ESG ---')
print(f"  ESG vs Age:    r = {df['ESG'].corr(df['Age']):.3f}")
print(f"  ESG vs Wealth: r = {df['ESG'].corr(df['Wealth']):.3f}")

### 2.7 EDA Summary

**Key takeaways from the exploratory analysis:**

- **Geographic disparities** exist in income and wealth across Italian macro-regions, but debt levels are more uniform
- **Investment behavior** is strongly linked to financial education and digital propensity — clients who invest in PAC plans are systematically more financially literate and digitally active
- **Older clients** tend toward lump-sum investments or no investments; capital accumulation (PAC) is more common among younger, employed clients
- **Wealthier and higher-saving clients** are more likely to invest, especially in PAC plans
- **Family size** shows a mild positive relationship with debt and a mild negative one with saving
- **ESG awareness** appears mildly positively correlated with age, challenging the assumption that only younger clients care about sustainability

These patterns will resurface in the cluster profiles: the clustering algorithm should capture these natural groupings without being explicitly told about them.